In [ ]:
# =========================
# LIMPIEZA DE ENTORNO
# =========================

!pip uninstall -y tensorflow tensorflow-cpu tensorflow-gpu protobuf transformers peft accelerate -q

# =========================
# INSTALACIÓN BASE (COMPATIBLE)
# =========================

!pip install transformers==4.38.2 accelerate==0.27.2 protobuf==3.20.3 -q

# =========================
# MÉTRICAS
# =========================

!pip install evaluate sacrebleu unbabel-comet -q

In [ ]:
import pandas as pd
from google.colab import drive
drive.mount('/content/drive')

path = "/content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Shipibo Konibo/train.csv"

df = pd.read_csv(path)
df = df[["source", "target"]].dropna()

In [ ]:
import re

def limpiar_texto(texto):
    return re.sub(r"[^\w\s-]", "", texto)

In [ ]:
#Aplicamos limpieza

df["source_clean"] = df["source"].apply(limpiar_texto)
df["target_clean"] = df["target"].apply(limpiar_texto)

# SILABIFICADOR

In [ ]:
VOWELS = {'a', 'e', 'i', 'o'}

DIGRAPHS = {'ch', 'sh', 'ts'}

CODAS = {'n', 's', 'sh', 'x'}

def tokenize(word):
    tokens = []
    i = 0

    while i < len(word):

        if i + 1 < len(word) and word[i:i+2] in DIGRAPHS:
            tokens.append(word[i:i+2])
            i += 2
        else:
            tokens.append(word[i])
            i += 1

    return tokens

def silabificador_shipibo(word):

    tokens = tokenize(word)

    syllables = []
    current = []

    i = 0

    while i < len(tokens):

        current.append(tokens[i])

        # si es vocal
        if tokens[i] in VOWELS:

            # mirar siguiente
            if i + 1 < len(tokens):

                nxt = tokens[i + 1]

                # caso VCV
                if nxt not in VOWELS:

                    # mirar siguiente del siguiente
                    if i + 2 < len(tokens):

                        nxt2 = tokens[i + 2]

                        # VCV → dividir antes de consonante
                        if nxt2 in VOWELS:
                            syllables.append(current)
                            current = []

                        # VCCV
                        else:

                            # coda válida
                            if nxt in CODAS:
                                current.append(nxt)
                                syllables.append(current)
                                current = []
                                i += 1
                            else:
                                syllables.append(current)
                                current = []

                    else:
                        # final
                        if nxt in CODAS:
                            current.append(nxt)
                            i += 1

            syllables.append(current)
            current = []

        i += 1

    if current:
        syllables.append(current)

    return [''.join(s) for s in syllables]

In [ ]:
def silabificar_texto(texto):
    """
    El silabificador trabaja palabra por palabra.

    Cada palabra:
    palabra → sílabas

    Luego todas las sílabas se unen
    en una sola secuencia.
    """

    palabras = texto.split()

    resultado = []

    for palabra in palabras:

        silabas = silabificador_shipibo(palabra)

        resultado.extend(silabas)

    return resultado

In [ ]:
df["source_silabificado"] = df["source_clean"].apply(
    lambda x: " ".join(silabificar_texto(x))
)

In [ ]:
for i in range(5):

    print("ORIGINAL:", df.iloc[i]["source"])

    print("LIMPIO:", df.iloc[i]["source_clean"])

    print("SILABIFICADO:", df.iloc[i]["source_silabificado"])

    print("TARGET:", df.iloc[i]["target_clean"])

    print("-----")

***Creación de dataset***

In [ ]:
from datasets import Dataset

df_model = df[["source_silabificado", "target_clean"]].rename(
    columns={
        "source_silabificado": "source",
        "target_clean": "target"
    }
)

dataset = Dataset.from_pandas(df_model)

print(dataset)

In [ ]:
print(dataset[0])

***Tokenización***

In [ ]:
##Llamar al tokenizador
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    "facebook/nllb-200-distilled-600M"
)

In [ ]:
tokenizer.src_lang = "quy_Latn"
tokenizer.tgt_lang = "spa_Latn"

In [ ]:
##Probar tokenización

example = dataset[0]["source"]
example2 = dataset[0]["target"]

print("Texto:")
print(example)

tokens = tokenizer(example)
tokens2 = tokenizer(example2)

print("\nTokens:")
print(tokens["input_ids"])
print(tokens2["input_ids"])

In [ ]:
## Función para tokenizar todo el dataset

def tokenizar_ejemplo(example):
    model_inputs = tokenizer(
        example["source"],
        max_length=64,
        truncation=True,
        padding="max_length"
    )

    labels = tokenizer(
        example["target"],
        max_length=64,
        truncation=True,
        padding="max_length"
    )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

In [ ]:
##Tokenizar todo el dataset

tokenized_dataset = dataset.map(tokenizar_ejemplo)

In [ ]:
##Limpiar el dataset de columnas iniciales

tokenized_dataset = tokenized_dataset.remove_columns(["source", "target"])
tokenized_dataset.set_format("torch")

In [ ]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Usando:", device)

***Modelo***

In [ ]:
##Llamar al modelo
from transformers import AutoModelForSeq2SeqLM

model = AutoModelForSeq2SeqLM.from_pretrained(
    "facebook/nllb-200-distilled-600M"
)

model.to(device)

In [ ]:
#Parámetros para el entrenamiento
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results_nllb",
    per_device_train_batch_size=1,
    num_train_epochs=6,
    learning_rate=3e-5,
    save_strategy="epoch",
    fp16=True
)

In [ ]:
##Creación del trainer

from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset
)

In [ ]:
#Entrenar el modelo
trainer.train()

In [ ]:
#Guardar el modelo

trainer.save_model("/content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Shipibo Konibo/modelo_NLLB_ShipiboKonibo_v1")
tokenizer.save_pretrained("/content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Shipibo Konibo/modelo_NLLB_ShipiboKonibo_v1")

In [ ]:
#Creación de la función de traducción
def traducir(texto):
    inputs = tokenizer(texto, return_tensors="pt").to(model.device)
    outputs = model.generate(**inputs, max_length=50)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# VALIDACIÓN (val.csv)

In [ ]:
#Cargar val

path_val = "/content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Shipibo Konibo/val.csv"

df_val = pd.read_csv(path_val)
df_val = df_val[["source", "target"]].dropna()

In [ ]:
#Preprocesamiento

df_val["source_clean"] = df_val["source"].apply(limpiar_texto)
df_val["target_clean"] = df_val["target"].apply(limpiar_texto)

df_val["source_silabificado"] = df_val["source_clean"].apply(
    lambda x: " ".join(silabificar_texto(x))
)

In [ ]:
#Ejemplos de los resultados

for i in range(5):
    print("INPUT:", df_val.iloc[i]["source_silabificado"])
    print("REAL:", df_val.iloc[i]["target_clean"])
    print("PRED:", traducir(df_val.iloc[i]["source_silabificado"]))
    print("-----")

# Métricas para val.csv

In [ ]:
#Preparar listas

preds = []
refs = []

for i in range(len(df_val)):
    pred = traducir(df_val.iloc[i]["source_silabificado"])
    ref = df_val.iloc[i]["target_clean"]

    preds.append(pred)
    refs.append([ref])

In [ ]:
#BLEU

import evaluate

bleu = evaluate.load("bleu")
print("BLEU:", bleu.compute(predictions=preds, references=refs))

In [ ]:
#CHRF

chrf = evaluate.load("chrf")
print("ChrF:", chrf.compute(predictions=preds, references=refs))

In [ ]:
#COMET

from comet import download_model, load_from_checkpoint

model_path = download_model("Unbabel/wmt20-comet-da")
comet_model = load_from_checkpoint(model_path)

data = []

for i in range(len(df_val)):
    data.append({
        "src": df_val.iloc[i]["source"],  # ORIGINAL
        "mt": preds[i],
        "ref": df_val.iloc[i]["target"]
    })

scores = comet_model.predict(data, batch_size=8, gpus=1)

print("COMET:", scores["system_score"])